# ЛР № 3. Профилирование и качество OWID Energy

Демонстрация участка `raw → staging`. Одна строка описывает одну страну за один год.
Вопрос из [Project Brief](../docs/project_brief.md): как изменяется углеродоёмкость электроэнергии
в европейских странах с разным преобладающим источником генерации с 2000 по 2025 год?
Бизнес-ключ `(country, year)`.

Запустите лабораторный стенд. Выберите ядро из `.venv` проекта.
Выполняйте ячейки сверху вниз. Для проверки повторяемости перезапустите ядро и выполните все ячейки ещё раз.

<a id="input"></a>
## Шаг 1. Входные данные и подключение

`ingested_on` - логическая дата загрузки, а не период поездок.
Аналитический период: 2000–2025 год по полю `year`.

Следующая ячейка определяет корень проекта при запуске из корня или каталога `notebooks`,
загружает настройки из `.env` без вывода секретов и использует подключение DuckDB из ЛР № 2.

In [1]:
import os
import sys
from html import escape
from pathlib import Path

import boto3
import duckdb
from dotenv import load_dotenv
from IPython.display import HTML, display

PROJECT_ROOT = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "pyproject.toml").is_file()
    and (parent / "src/analysis/read_raw.py").is_file()
)
load_dotenv(PROJECT_ROOT / ".env", override=True)
required = ("S3_ENDPOINT", "S3_ACCESS_KEY", "S3_SECRET_KEY")
missing = [name for name in required if not os.environ.get(name)]
if missing:
    raise ValueError("Не заданы настройки: " + ", ".join(missing))

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from analysis.read_raw import configure_minio  # noqa: E402

raw_path = "s3://raw/owid_energy/ingested_on=2026-10-02/owid-energy-data.csv"
period_start = 2000
period_end = 2025
output_prefix = "s3://staging/owid_energy/source_ingested_on=2026-10-02"

europe_iso3 = [
    "ALB","AUT","BEL","BIH","BGR","HRV","CYP","CZE","DNK","EST","FIN","FRA",
    "DEU","GRC","HUN","ISL","IRL","ITA","XKX","LVA","LIE","LTU","LUX","MLT",
    "MDA","MNE","NLD","MKD","NOR","POL","PRT","ROU","SRB","SVK","SVN","ESP",
    "SWE","CHE","GBR",
]

def literal(value):
    return "'" + value.replace("'", "''") + "'"

def show_table(headers, rows):
    header = "".join(f"<th>{escape(str(item))}</th>" for item in headers)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(str(item))}</td>" for item in row) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))

connection = duckdb.connect()
configure_minio(connection)
connection.execute(
    f"CREATE OR REPLACE TEMP VIEW raw_data AS SELECT * FROM read_csv_auto({literal(raw_path)})"
)
show_table(["Вход", "Начало периода", "Конец периода", "Стран в периметре"], [(raw_path, period_start, period_end, len(europe_iso3))])
show_table(["Страна", "ISO3", "Год", "Углеродоёмкость", "Доля ВИЭ"], connection.execute("""
    SELECT country, iso_code, year, carbon_intensity_elec, renewables_share_elec
        FROM raw_data LIMIT 3
    """).fetchall())

Вход,Начало периода,Конец периода,Стран в периметре
s3://raw/owid_energy/ingested_on=2026-10-02/owid-energy-data.csv,2000,2025,39


Страна,ISO3,Год,Углеродоёмкость,Доля ВИЭ
ASEAN (Ember),None,2000,572.58,19.334
ASEAN (Ember),None,2001,570.27,19.055
ASEAN (Ember),None,2002,572.68,17.667


<a id="profile"></a>
## Шаг 2. Профиль исходных данных

Выбраны пять полей для профиля: country и year (составной бизнес-ключ для группировки), carbon_intensity_elec (углеродоёмкость для основного вопроса), renewables_share_elec (доля ВИЭ для уточняющего вопроса) и три представителя долей генерации - coal_share_elec, gas_share_elec, nuclear_share_elec

Профиль рассчитывается на сыром датасете. При чтении из S3 фиксируется 23 377 строк и 131 поле. При этом DuckDB автоматически получает
дополнительное поле `ingested_on` из S3-пути.

In [2]:
raw_count = connection.execute("SELECT count(*) FROM raw_data").fetchone()[0]
schema = {row[0]: row[1] for row in connection.execute("DESCRIBE raw_data").fetchall()}
show_table(["Строк", "Полей при чтении S3"], [(raw_count, len(schema))])

Строк,Полей при чтении S3
23377,131


In [3]:
selected_fields = (
    "country", "year",
    "carbon_intensity_elec", "renewables_share_elec",
    "coal_share_elec", "gas_share_elec", "nuclear_share_elec",
)

null_meaning = {
    "country": "строки-агрегаты без привязки к стране",
    "year": "запись без года",
    "carbon_intensity_elec": "нет расчёта углеродоёмкости для страны/года",
    "renewables_share_elec": "нет данных по генерации ВИЭ за год",
    "coal_share_elec": "нет данных по угольной генерации за год",
    "gas_share_elec": "нет данных по газовой генерации за год",
    "nuclear_share_elec": "нет данных по атомной генерации за год",
}

profile_rows = []
for field in selected_fields:
    missing_count = connection.execute(
        f"SELECT count(*) FROM raw_data WHERE {field} IS NULL"
    ).fetchone()[0]
    examples = connection.execute(
        f"SELECT DISTINCT {field} FROM raw_data WHERE {field} IS NOT NULL ORDER BY 1 LIMIT 2"
    ).fetchall()
    share = f"{missing_count / raw_count:.3%}" if raw_count else "N/A"
    profile_rows.append((
        field,
        schema[field],
        ", ".join(str(row[0]) for row in examples),
        raw_count,
        missing_count,
        share,
        null_meaning.get(field, "—"),
    ))

show_table(["Поле", "Тип", "Примеры", "Проверено строк", "Пропусков", "Доля", "Смысл пропуска"], profile_rows)

Поле,Тип,Примеры,Проверено строк,Пропусков,Доля,Смысл пропуска
country,VARCHAR,"ASEAN (Ember), Afghanistan",23377,0,0.000%,строки-агрегаты без привязки к стране
year,BIGINT,"1900, 1901",23377,0,0.000%,запись без года
carbon_intensity_elec,DOUBLE,"0.0, 17.24",23377,17045,72.914%,нет расчёта углеродоёмкости для страны/года
renewables_share_elec,DOUBLE,"0.0, 0.004",23377,15505,66.326%,нет данных по генерации ВИЭ за год
coal_share_elec,DOUBLE,"0.0, 0.017",23377,16235,69.449%,нет данных по угольной генерации за год
gas_share_elec,DOUBLE,"0.0, 0.002",23377,16378,70.060%,нет данных по газовой генерации за год
nuclear_share_elec,DOUBLE,"0.0, 0.012",23377,15659,66.985%,нет данных по атомной генерации за год


In [4]:
period = connection.execute(
    "SELECT min(year), max(year) FROM raw_data"
).fetchone()

exact_duplicates = connection.execute("""
    SELECT count(*) - (SELECT count(*) FROM (SELECT DISTINCT * FROM raw_data))
    FROM raw_data
""").fetchone()[0]

dup_key = connection.execute("""
    SELECT count(*)
    FROM (
        SELECT country, year
        FROM raw_data
        GROUP BY country, year
        HAVING count(*) > 1
    )
""").fetchone()[0]

show_table(["Минимальный год", "Максимальный год", "Полных повторов", "Повторы по country+year"],
           [(*period, exact_duplicates, dup_key)])

years = connection.execute(f"""
    SELECT year, count(*) AS rows
    FROM raw_data
    WHERE year BETWEEN {period_start} AND {period_end}
    GROUP BY 1 ORDER BY 1
""").fetchall()
show_table(["Год", "Записей"], years)


Минимальный год,Максимальный год,Полных повторов,Повторы по country+year
1900,2025,0,0


Год,Записей
2000,306
2001,306
2002,306
2003,307
2004,307
2005,308
2006,307
2007,307
2008,307
2009,307


### Наблюдения для отчёта

- В исходном наборе доля пропусков по метрикам (`carbon_intensity_elec`, `*_share_elec`) составляет от 66% до 73%. Как видно из примеров поля country (например, ASEAN (Ember)), это связано с тем, что OWID Energy включает данные по всему миру, включая регионы и страны, где электроэнергетика не развита или отсутствует отчётность.
- Фактический диапазон дат в raw-данных - с 1900 по 2025 год. Однако распределение по годам показывает, что в 2024 году число записей падает до 254, а в 2025 — до 106 (на фоне стабильных 300 записей в предыдущие годы). 2025 год содержит неполные данные. При анализе динамики до 2025 года это может привести к ложным выводам о резком падении углеродоёмкости или доли ВИЭ из-за простого отсутствия части стран в датасете. 
- Проверка бизнес-ключа country + year показала 0 повторов и 0 полных дублей строк. Это значит, что ключ уникален. Способ подсчёта: группировка данных по country и year с фильтром HAVING count(*) > 1. Дополнительно проверены полные дубли строк (расчёт как общее число строк минус число уникальных строк через DISTINCT) - также обнаружено 0 совпадений.

<a id="rules"></a>
## Шаг 3. Пять конкретных правил качества

Для каждого правила ниже указаны идентификатор, поля, проверяемые строки, условие, критичность,
действие и связь с задачей. Порог задан переменной в ячейке и участвует в сравнении.
В демонстрации единица порога - число нарушений, допустимый максимум - ноль.

Порог оценивает набор. Действие по каждой нарушившей строке применяется независимо от превышения порога.
Доля считается внутри проверяемой совокупности; при пустой совокупности доля и сравнение - `N/A`.

In [5]:
rule_results = {}

def show_rule(rule_id, checked, violations, maximum, action):
    share = f"{violations / checked:.3%}" if checked else "N/A"
    comparison = ("Превышен" if violations > maximum else "Не превышен") if checked else "N/A"
    row = (rule_id, checked, violations, share, f"count ≤ {maximum}", comparison, action)
    rule_results[rule_id] = row
    show_table(["Правило", "Проверено", "Нарушений", "Доля", "Порог", "Сравнение", "Действие"],
               [row])

### Правило 1. `business_key_unique`

- Поля: `country`, `year`.
- Проверяемые строки: все строки в raw_data.
- Условие: комбинация country + year встречается ровно один раз.
- Порог: число нарушений; переменная `max_duplicates` в следующей ячейке.
- Критичность: `critical`.
- Действие: `quarantine`.
- Связь с задачей: без уникальности ключа нельзя корректно рассчитать средние и медианы по странам и годам.

In [6]:
max_duplicates = 0
checked, violations = connection.execute("""
    SELECT 
        (SELECT count(*) FROM raw_data),
        count(*)
    FROM (
        SELECT country, year
        FROM raw_data
        GROUP BY country, year
        HAVING count(*) > 1
    )
""").fetchone()

show_rule("business_key_unique", checked, violations, max_duplicates, "quarantine")

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Действие
business_key_unique,23377,0,0.000%,count ≤ 0,Не превышен,quarantine


### Правило 2. `carbon_intensity_non_negative`

- Поля: `carbon_intensity_elec`.
- Проверяемые строки: все строки, где `carbon_intensity_elec` не пуст.
- Условие: `carbon_intensity_elec` >= 0.
- Порог: число нарушений; переменная `max_negative_ci`.
- Критичность: `critical`.
- Действие: `quarantine`.
- Связь с задачей: отрицательные значения физически невозможны и исказят расчет средних значений при ответе на основной вопрос.


In [7]:
max_negative_ci = 0

checked, violations = connection.execute("""
    SELECT 
        count(*) FILTER (WHERE carbon_intensity_elec IS NOT NULL), 
        count(*) FILTER (WHERE carbon_intensity_elec < 0) 
    FROM raw_data
""").fetchone()

show_rule("carbon_intensity_non_negative", checked, violations, max_negative_ci, "quarantine")

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Действие
carbon_intensity_non_negative,6332,0,0.000%,count ≤ 0,Не превышен,quarantine


### Правило 3. `shares_within_bounds`

- Поле: `*_share_elec`.
- Проверяемые строки: строки, где хотя бы одна из долей присутствует.
- Условие: значение от 0 до 100.
- Порог: число нарушений; переменная `max_invalid_shares`.
- Критичность: `critical`.
- Действие: `quarantine`.
- Связь с задачей: значения за пределами 0–100% исказят медиану доли ВИЭ и сломают логику определения преобладающего источника.

In [8]:
max_invalid_shares = 0

checked, violations = connection.execute("""
    SELECT 
        count(*) FILTER (WHERE renewables_share_elec IS NOT NULL OR coal_share_elec IS NOT NULL),
        count(*) FILTER (
            WHERE renewables_share_elec < 0 OR renewables_share_elec > 100
               OR coal_share_elec < 0 OR coal_share_elec > 100
               OR gas_share_elec < 0 OR gas_share_elec > 100
               OR nuclear_share_elec < 0 OR nuclear_share_elec > 100
        )
    FROM raw_data
""").fetchone()

show_rule("shares_within_bounds", checked, violations, max_invalid_shares, "quarantine")

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Действие
shares_within_bounds,7887,0,0.000%,count ≤ 0,Не превышен,quarantine


### Правило 4. `target_period_validity`

- Поле: `year`.
- Проверяемые строки: все строки.
- Условие: 2000 до 2025.
- Порог: число нарушений; переменная `max_out_of_period`.
- Критичность: `critical`.
- Действие: `quarantine`.
- Связь с задачей: фокус исследования ограничен 2000–2025 годами, данные за 1900–1999 годы не нужны и могут исказить динамику.

Правило показывает масштаб отсечения.

In [9]:
max_out_of_period = 0

checked, violations = connection.execute("""
    SELECT 
        count(*), 
        count(*) FILTER (WHERE year < 2000 OR year > 2025) 
    FROM raw_data
""").fetchone()

show_rule("target_period_validity", checked, violations, max_out_of_period, "quarantine")

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Действие
target_period_validity,23377,15741,67.335%,count ≤ 0,Превышен,quarantine


### Правило 5. `shares_sum_not_exceed_100`

- Поля: поля `*_share_elec`.
- Проверяемые строки: страны и годы, где хотя бы одно поле доли заполнено.
- Условие: cумма долей <= 100.
- Порог: число нарушений; переменная `max_shares_sum`.
- Критичность: `critical`.
- Действие: `quarantine`.
- Связь с задачей: если сумма долей превышает 100%, расчет медианной доли ВИЭ и определение преобладающего источника будут некорректными.


In [10]:
max_shares_sum = 0

checked, violations = connection.execute("""
    SELECT 
        count(*) FILTER (
            WHERE coal_share_elec IS NOT NULL 
               OR gas_share_elec IS NOT NULL 
               OR renewables_share_elec IS NOT NULL
        ),
        count(*) FILTER (
            WHERE coalesce(coal_share_elec, 0) + 
                  coalesce(gas_share_elec, 0) + 
                  coalesce(nuclear_share_elec, 0) + 
                  coalesce(hydro_share_elec, 0) + 
                  coalesce(solar_share_elec, 0) + 
                  coalesce(wind_share_elec, 0) + 
                  coalesce(biofuel_share_elec, 0) > 100.01
        )
    FROM raw_data
""").fetchone()

show_rule("shares_sum_not_exceed_100", checked, violations, max_shares_sum, "quarantine")

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Действие
shares_sum_not_exceed_100,7887,0,0.000%,count ≤ 0,Не превышен,quarantine


<a id="prepare"></a>
## Шаг 4. Staging и карантин

Имена и типы выбранных аналитических полей уже подходят - искусственно менять их не нужно.
В staging остаются строки без критических нарушений; отрицательная сумма отмечается Boolean-признаком.
В карантине исходные поля сохраняются вместе со всеми критическими причинами через `;`.
При нескольких причинах запись сохраняется один раз. Карантин имеет приоритет перед флагом.

`COALESCE(..., false)` ниже относится к результату сравнения, а не заменяет исходное значение нулём.

In [11]:
connection.execute("""
    CREATE OR REPLACE TEMP VIEW checked_data AS
    SELECT *,
        -- Правило 1: Дубликаты по бизнес-ключу (country + year)
        count(*) OVER (PARTITION BY country, year) > 1 AS duplicate_key,
        
        -- Правило 2: Отрицательная углеродоёмкость
        COALESCE(carbon_intensity_elec < 0, false) AS negative_carbon_intensity,
        
        -- Правило 3: Доли источников вне диапазона 0-100%
        COALESCE(
            renewables_share_elec < 0 OR renewables_share_elec > 100 OR
            coal_share_elec < 0 OR coal_share_elec > 100 OR
            gas_share_elec < 0 OR gas_share_elec > 100 OR
            nuclear_share_elec < 0 OR nuclear_share_elec > 100 OR
            hydro_share_elec < 0 OR hydro_share_elec > 100 OR
            solar_share_elec < 0 OR solar_share_elec > 100 OR
            wind_share_elec < 0 OR wind_share_elec > 100 OR
            biofuel_share_elec < 0 OR biofuel_share_elec > 100
        , false) AS invalid_shares,
        
        -- Правило 4: Год вне периода 2000-2025
        COALESCE(year < 2000 OR year > 2025, false) AS outside_period,
        
        -- Правило 5: Сумма долей превышает 100% (допускаем микроскопическую погрешность float)
        COALESCE(
            coalesce(coal_share_elec, 0) + coalesce(gas_share_elec, 0) + coalesce(nuclear_share_elec, 0) +
            coalesce(hydro_share_elec, 0) + coalesce(solar_share_elec, 0) + coalesce(wind_share_elec, 0) +
            coalesce(biofuel_share_elec, 0) > 100.01
        , false) AS shares_sum_exceeded
    FROM raw_data
""")

connection.execute("""
    CREATE OR REPLACE TEMP VIEW staging_data AS
    SELECT * EXCLUDE (duplicate_key, negative_carbon_intensity, invalid_shares, outside_period, shares_sum_exceeded),
        carbon_intensity_elec IS NULL AS quality_carbon_intensity_missing
    FROM checked_data
    WHERE NOT (duplicate_key OR negative_carbon_intensity OR invalid_shares OR outside_period OR shares_sum_exceeded)
""")

connection.execute("""
    CREATE OR REPLACE TEMP VIEW quarantine_data AS
    SELECT * EXCLUDE (duplicate_key, negative_carbon_intensity, invalid_shares, outside_period, shares_sum_exceeded),
        concat(
            CASE WHEN duplicate_key THEN 'business_key_unique;' ELSE '' END,
            CASE WHEN negative_carbon_intensity THEN 'carbon_intensity_non_negative;' ELSE '' END,
            CASE WHEN invalid_shares THEN 'shares_within_bounds;' ELSE '' END,
            CASE WHEN outside_period THEN 'target_period_validity;' ELSE '' END,
            CASE WHEN shares_sum_exceeded THEN 'shares_sum_not_exceed_100;' ELSE '' END
        ) AS quality_rule_ids,
        concat(
            CASE WHEN duplicate_key THEN 'Дубликат ключа страна+год; ' ELSE '' END,
            CASE WHEN negative_carbon_intensity THEN 'Отрицательная углеродоёмкость; ' ELSE '' END,
            CASE WHEN invalid_shares THEN 'Доли вне 0-100%; ' ELSE '' END,
            CASE WHEN outside_period THEN 'Год вне 2000-2025; ' ELSE '' END,
            CASE WHEN shares_sum_exceeded THEN 'Сумма долей > 100%; ' ELSE '' END
        ) AS quarantine_reason
    FROM checked_data
    WHERE duplicate_key OR negative_carbon_intensity OR invalid_shares OR outside_period OR shares_sum_exceeded
""")

show_table(["Набор", "Строк"], [
    (name, connection.execute(f"SELECT count(*) FROM {name}_data").fetchone()[0])
    for name in ("raw", "staging", "quarantine")
])

Набор,Строк
raw,23377
staging,7636
quarantine,15741


In [12]:
examples = connection.execute("""
    (SELECT country, year, carbon_intensity_elec, renewables_share_elec, 
            quality_rule_ids, quarantine_reason FROM quarantine_data
     WHERE quality_rule_ids LIKE '%target_period_validity%'
     ORDER BY year LIMIT 1)
    UNION ALL
    (SELECT country, year, carbon_intensity_elec, renewables_share_elec, 
            quality_rule_ids, quarantine_reason FROM quarantine_data
     WHERE quality_rule_ids LIKE '%shares_sum_not_exceed_100%'
     ORDER BY year LIMIT 1)
""").fetchall()

show_table(["Страна", "Год", "Углеродоёмкость", "Доля ВИЭ", "Правила", "Причина"], examples)

Страна,Год,Углеродоёмкость,Доля ВИЭ,Правила,Причина
Africa,1900,None,None,target_period_validity;,Год вне 2000-2025;


### Запись двух результатов в S3

Следующая ячейка заменяет только `staging.parquet` и `quarantine.parquet`.

In [13]:
source_date = raw_path.split("/ingested_on=", 1)[1].split("/", 1)[0]
expected_prefix = f"s3://staging/owid_energy/source_ingested_on={source_date}"
if output_prefix != expected_prefix:
    raise ValueError("Выходной префикс не соответствует исходному снимку.")
endpoint = os.environ["S3_ENDPOINT"]
client = boto3.client(
    "s3", endpoint_url=endpoint if endpoint.startswith("http") else f"http://{endpoint}",
    aws_access_key_id=os.environ["S3_ACCESS_KEY"],
    aws_secret_access_key=os.environ["S3_SECRET_KEY"], region_name="us-east-1",
)
prefix = output_prefix.removeprefix("s3://staging/")
for name in ("staging", "quarantine"):
    key = f"{prefix}/{name}.parquet"
    client.delete_object(Bucket="staging", Key=key)
    connection.execute(
        f"COPY (SELECT * FROM {name}_data) TO {literal('s3://staging/' + key)} (FORMAT PARQUET)"
    )
show_table(["Результат", "S3-путь"], [
    (name, f"{output_prefix}/{name}.parquet") for name in ("staging", "quarantine")
])

Результат,S3-путь
staging,s3://staging/owid_energy/source_ingested_on=2026-10-02/staging.parquet
quarantine,s3://staging/owid_energy/source_ingested_on=2026-10-02/quarantine.parquet


<a id="report"></a>
## Шаг 5. Краткий отчёт

Таблицы и значимые наблюдения в [docs/data_quality_report.md](../docs/data_quality_report.md).
Отчёт содержит пять разделов: контекст, профиль, правила и результаты, подготовленные данные,
воспроизводимость и ограничения.

<a id="verify"></a>
## Шаг 6. Проверка сохранённых файлов

Читаем оба файла непосредственно из S3.
Проверяем баланс, отсутствие критических нарушений, правильность флага и причины карантина.
Количество полей при чтении может включать `source_ingested_on` из выходного пути -
его нет внутри Parquet-файлов.

In [14]:
for name in ("staging", "quarantine"):
    path = f"{output_prefix}/{name}.parquet"
    connection.execute(
        f"CREATE OR REPLACE TEMP VIEW saved_{name} AS SELECT * FROM read_parquet({literal(path)})"
    )

staging_count = connection.execute("SELECT count(*) FROM saved_staging").fetchone()[0]
quarantine_count = connection.execute("SELECT count(*) FROM saved_quarantine").fetchone()[0]

critical_count = connection.execute("""
    SELECT count(*) FROM saved_staging
    WHERE COALESCE(carbon_intensity_elec < 0, false)
       OR COALESCE(year < 2000 OR year > 2025, false)
       OR COALESCE(renewables_share_elec < 0 OR renewables_share_elec > 100, false)
       OR COALESCE(coal_share_elec < 0 OR coal_share_elec > 100, false)
       OR COALESCE(gas_share_elec < 0 OR gas_share_elec > 100, false)
       -- (можно добавить остальные доли, но для проверки хватит и этих)
""").fetchone()[0]

bad_flags = connection.execute("""
    SELECT count(*) FROM saved_staging
    WHERE quality_carbon_intensity_missing
          IS DISTINCT FROM (carbon_intensity_elec IS NULL)
""").fetchone()[0]

flagged_count = connection.execute("""
    SELECT count(*) FROM saved_staging
    WHERE quality_carbon_intensity_missing
""").fetchone()[0]

bad_reasons = connection.execute("""
    SELECT count(*) FROM saved_quarantine
    WHERE quality_rule_ids IS NULL OR quality_rule_ids = ''
       OR quarantine_reason IS NULL OR quarantine_reason = ''
""").fetchone()[0]

assert raw_count == staging_count + quarantine_count, "Не сошёлся баланс строк (Raw != Staging + Quarantine)."
assert critical_count == 0, "В staging попали критические нарушения!"
assert bad_flags == 0, "Некорректные признаки пропуска углеродоёмкости."
assert bad_reasons == 0, "В карантине есть строки без указания причины."

run_summary = (raw_count, staging_count, quarantine_count, flagged_count, critical_count)
show_table(["Raw", "Staging", "Карантин", "С флагом", "Крит. нарушения"], [run_summary])
show_table(["Проверка", "Нарушений"], [
    ("Неверные флаги", bad_flags), 
    ("Пустые причины в карантине", bad_reasons)
])

Raw,Staging,Карантин,С флагом,Крит. нарушения
23377,7636,15741,1634,0


Проверка,Нарушений
Неверные флаги,0
Пустые причины в карантине,0


In [15]:
show_table(["Страна", "Год", "Углеродоёмкость", "Флаг пропуска"], connection.execute("""
    SELECT country, year, carbon_intensity_elec, quality_carbon_intensity_missing
    FROM saved_staging 
    WHERE quality_carbon_intensity_missing
    ORDER BY year DESC, country 
    LIMIT 3
""").fetchall())
show_table(["Страна", "Год", "Правила", "Причина"], connection.execute("""
    SELECT country, year, quality_rule_ids, quarantine_reason
    FROM saved_quarantine 
    ORDER BY year DESC, country 
    LIMIT 3
""").fetchall())

Страна,Год,Углеродоёмкость,Флаг пропуска
Africa (EI),2024,None,True
Asia Pacific (EI),2024,None,True
CIS (EI),2024,None,True


Страна,Год,Правила,Причина
Afghanistan,1999,target_period_validity;,Год вне 2000-2025;
Africa,1999,target_period_validity;,Год вне 2000-2025;
Africa (EI),1999,target_period_validity;,Год вне 2000-2025;
